# Shared utility functions for the Bronze -> Silver layer.
 Usage in another notebook:
-  %run ./silver_common

In [0]:




from pyspark.sql import DataFrame, functions as F
from pyspark.sql.window import Window
from datetime import datetime

CATALOG = "Catalog_name"      # confirmed via SHOW CATALOGS
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
DQ_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.dq_log"

CROSSWALK_MODE = "lakehouse_federation"   
AZURE_SQL_CATALOG = "azure_sql"          
FUND_XWALK_TABLE = f"{AZURE_SQL_CATALOG}.staging.fund_id_mapping"
ASSET_XWALK_TABLE = f"{AZURE_SQL_CATALOG}.staging.asset_id_mapping"

# --- Option B: direct JDBC (use if Lakehouse Federation isn't set up) ---
JDBC_URL = "jdbc:sqlserver://<your-server>.database.windows.net:1433;database=<your-db>"
JDBC_SECRET_SCOPE = "pe-fund-secrets"     # <-- CHANGE: Databricks secret scope holding SQL creds


def bronze_table(name: str) -> str:
    """Fully-qualified Bronze table name."""
    return f"{CATALOG}.{BRONZE_SCHEMA}.{name}"


def silver_table(name: str) -> str:
    """Fully-qualified Silver table name."""
    return f"{CATALOG}.{SILVER_SCHEMA}.{name}"


def read_crosswalk(spark, which: str) -> DataFrame:
    """
    Read a crosswalk table. which = 'fund' or 'asset'.
    Returns a DataFrame with the mapping table's native columns -
    caller renames as needed (column names weren't given in the handoff
    note, so confirm exact column names once you can query the table:
    e.g. `SELECT * FROM staging.fund_id_mapping` in Azure SQL, or
    `SHOW COLUMNS IN <catalog>.staging.fund_id_mapping` if federated).
    """
    table_name = FUND_XWALK_TABLE if which == "fund" else ASSET_XWALK_TABLE

    if CROSSWALK_MODE == "lakehouse_federation":
        return spark.table(table_name)

    elif CROSSWALK_MODE == "jdbc":
        schema, tbl = table_name.split(".", 1) if "." in table_name else ("staging", table_name)
        user = dbutils.secrets.get(scope=JDBC_SECRET_SCOPE, key="sql-user")
        pwd = dbutils.secrets.get(scope=JDBC_SECRET_SCOPE, key="sql-password")
        return (
            spark.read.format("jdbc")
            .option("url", JDBC_URL)
            .option("dbtable", table_name)
            .option("user", user)
            .option("password", pwd)
            .load()
        )
    else:
        raise ValueError(f"Unknown CROSSWALK_MODE: {CROSSWALK_MODE}")


def read_bronze(spark, name: str) -> DataFrame:
    """Read a Bronze Delta table by short name, e.g. read_bronze(spark, 'fund')."""
    return spark.table(bronze_table(name))



# Null handling

def split_on_required_nulls(df: DataFrame, required_cols: list) -> tuple:
    """
    Splits a DataFrame into (clean_df, rejected_df) based on required_cols.
    Any row with a null in ANY required column goes to rejected_df, tagged
    with a reason_code column. clean_df keeps every row, nulls in optional
    columns included (that's expected, not an error - see Section 1.1 of
    the DE-2 doc: optional-field nulls are not rejected).
    """
    null_condition = None
    for c in required_cols:
        cond = F.col(c).isNull()
        null_condition = cond if null_condition is None else (null_condition | cond)

    rejected_df = (
        df.filter(null_condition)
          .withColumn("reason_code", F.lit("NULL_REQUIRED_FIELD"))
          .withColumn("failed_columns", F.array(*[
              F.when(F.col(c).isNull(), F.lit(c)) for c in required_cols
          ]))
          .withColumn("failed_columns", F.array_except("failed_columns", F.array(F.lit(None).cast("string"))))
    )
    clean_df = df.filter(~null_condition)
    return clean_df, rejected_df



# Duplicate / break detection

def split_duplicates(df: DataFrame, key_cols: list, compare_cols: list) -> tuple:
    """
    Splits a DataFrame into (clean_df, duplicates_df, breaks_df) using a
    business key (key_cols) and a set of value columns to compare
    (compare_cols).

    - Exact repeat: same key_cols AND same compare_cols values -> duplicate,
      only the first occurrence is kept in clean_df, the rest go to
      duplicates_df tagged DUPLICATE_RECORD.
    - Same key_cols, DIFFERENT compare_cols values -> both/all versions kept
      in clean_df (never silently overwritten) AND also written to
      breaks_df tagged with reason_code '<CONTEXT>_BREAK' for the
      reconciliation team to investigate. Caller sets break_reason_code.
    """
    w = Window.partitionBy(*key_cols).orderBy(*compare_cols)
    df2 = df.withColumn("_compare_hash", F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in compare_cols]), 256))

    distinct_vals_per_key = df2.groupBy(*key_cols).agg(F.countDistinct("_compare_hash").alias("_distinct_val_count"))
    df3 = df2.join(distinct_vals_per_key, on=key_cols, how="left")

    w_exact = Window.partitionBy(*key_cols, "_compare_hash").orderBy(F.lit(1))
    df4 = df3.withColumn("_rn_exact", F.row_number().over(w_exact))

    duplicates_df = (
        df4.filter(F.col("_rn_exact") > 1)
           .withColumn("reason_code", F.lit("DUPLICATE_RECORD"))
           .drop("_compare_hash", "_distinct_val_count", "_rn_exact")
    )

    breaks_df = (
        df4.filter(F.col("_distinct_val_count") > 1)
           .drop("_compare_hash", "_distinct_val_count", "_rn_exact")
    )

    clean_df = (
        df4.filter(F.col("_rn_exact") == 1)   # drop exact-duplicate extras
           .drop("_compare_hash", "_distinct_val_count", "_rn_exact")
    )

    return clean_df, duplicates_df, breaks_df



# DQ logging - every rule failure across every source lands in one table

def log_dq(spark, source_name: str, business_date: str, rule_name: str,
           records_checked: int, records_failed: int, reason_code: str):
    """Append one summary row to the shared DQ log table."""
    row = spark.createDataFrame([{
        "source_name": source_name,
        "business_date": business_date,
        "rule_name": rule_name,
        "records_checked": records_checked,
        "records_failed": records_failed,
        "reason_code": reason_code,
        "created_at": datetime.utcnow().isoformat(),
    }])
    row.write.format("delta").mode("append").saveAsTable(DQ_TABLE)



# Silver container root 

SILVER_CONTAINER_URL = "abfss://silver@pestorage3.dfs.core.windows.net"  


def write_quarantine(df: DataFrame, source_name: str):
    """
    Append rejected/duplicate/break rows to a per-source Silver quarantine
    table, stored at a named path (silver/_quarantine/<source_name>/) so it's
    visible in Storage Explorer the same way Bronze's _quarantine/ folder is.
    """
    table_name = f"_quarantine_{source_name}"
    path = f"{SILVER_CONTAINER_URL}/_quarantine/{source_name}/"
    (df.withColumn("quarantined_at", F.current_timestamp())
       .write.format("delta").mode("append")
       .option("mergeSchema", "true")
       .save(path))
    spark.sql(f"CREATE TABLE IF NOT EXISTS {silver_table(table_name)} USING DELTA LOCATION '{path}'")


def write_silver(df: DataFrame, name: str, mode: str = "overwrite"):
    """
    Write the clean DataFrame to its Silver Delta table at a named path
    (silver/<name>/) so it shows up in Storage Explorer the same way Bronze's
    fund/, investor/, etc. folders do - registers as an external table on
    first write, and plain overwrites/appends the same path on every run
    after that.
    """
    path = f"{SILVER_CONTAINER_URL}/{name}/"
    df.write.format("delta").mode(mode).option("mergeSchema", "true").save(path)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {silver_table(name)} USING DELTA LOCATION '{path}'")


# Gold container root 

GOLD_SCHEMA = "gold"
GOLD_CONTAINER_URL = "abfss://gold@pestorage3.dfs.core.windows.net"  


def gold_table(name: str) -> str:
    """Fully-qualified Gold table name."""
    return f"{CATALOG}.{GOLD_SCHEMA}.{name}"


def write_gold(df: DataFrame, name: str, mode: str = "overwrite"):
    """
    Write a DataFrame to its Gold Delta table at a named path (gold/<name>/),
    mirroring write_silver()'s convention exactly - registers as an external
    table on first write, plain overwrites/appends the same path after that.
    """
    path = f"{GOLD_CONTAINER_URL}/{name}/"
    df.write.format("delta").mode(mode).option("mergeSchema", "true").save(path)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {gold_table(name)} USING DELTA LOCATION '{path}'")


# Referential integrity

def flag_late(df: DataFrame, ts_col: str, business_date_col: str, cutoff_hour: int = 12) -> DataFrame:
    """
    Adds an `is_late` boolean column: True if ts_col's time-of-day is after
    cutoff_hour (24h, local to the timestamp as stored) relative to its own
    business_date_col. Used for CASH's Day-3 late record (21:45 vs ~11:00
    normal batch time) - default cutoff of 12:00 comfortably separates the
    two. Adjust cutoff_hour once the team agrees the exact rule (see
    Section 1.3 CASH note - "agree with the mentor before coding").
    """
    return df.withColumn(
        "is_late",
        F.when(F.hour(F.col(ts_col)) >= cutoff_hour, F.lit(True)).otherwise(F.lit(False))
    )


def flag_day_over_day_change(df: DataFrame, key_cols: list, business_date_col: str, compare_cols: list) -> DataFrame:
    """
    For each key (e.g. asset_id), compares each row's compare_cols values to
    the PREVIOUS business_date's row for the same key. Adds:
      - `prev_values`: struct of the previous day's compare_cols
      - `changed`: True if any compare_col differs from the previous day
    Used for REFERENCE's currency-flip detection (Section 1.3: AST005
    EUR -> GBP -> EUR). First-ever row for a key has changed = False
    (nothing to compare against yet) - that's correct, not a bug.
    """
    w = Window.partitionBy(*key_cols).orderBy(business_date_col)

    df2 = df
    for c in compare_cols:
        df2 = df2.withColumn(f"_prev_{c}", F.lag(F.col(c)).over(w))

    changed_condition = None
    for c in compare_cols:
        cond = (F.col(c) != F.col(f"_prev_{c}")) & F.col(f"_prev_{c}").isNotNull()
        changed_condition = cond if changed_condition is None else (changed_condition | cond)

    df2 = df2.withColumn("changed", changed_condition)
    return df2


def find_business_key_duplicates(df: DataFrame, business_key_cols: list, tie_break_cols: list) -> tuple:
    """
    Catches duplicates that share identical BUSINESS values but have
    DIFFERENT id/primary-key values - the case split_duplicates() cannot
    catch, since it keys on a single id column. Example: PMT-DUP-20260917-001
    and PMT00220260917 have different payment_ids but identical fund_id,
    payment_type, amount, and event_timestamp - a genuine duplicate under a
    different name.

    tie_break_cols: list of columns to sort by (ascending) to decide which
    row survives per business-key group - the FIRST row after sorting by
    all of these, in order, wins. Pass more than one column when a single
    column's natural sort order could pick a synthetic/test-labeled id over
    the "real" one (e.g. a computed priority column should come before the
    raw id column - see 10_silver_external_payment for an example).
    """
    w = Window.partitionBy(*business_key_cols).orderBy(*[F.col(c).asc() for c in tie_break_cols])
    ranked = df.withColumn("_bk_rank", F.row_number().over(w))

    duplicates_df = (
        ranked.filter(F.col("_bk_rank") > 1)
              .drop("_bk_rank")
              .withColumn("reason_code", F.lit("BUSINESS_KEY_DUPLICATE"))
    )
    clean_df = ranked.filter(F.col("_bk_rank") == 1).drop("_bk_rank")
    return clean_df, duplicates_df


def merge_into_silver(df: DataFrame, name: str, merge_key_cols: list):
    """
    Delta MERGE (upsert) into a Silver table, keyed on merge_key_cols -
    the Day-7 incremental-processing primitive. Unlike write_silver()
    (which overwrites the whole table), this updates matching rows and
    inserts new ones, so re-running the same day's data twice never
    duplicates it. Creates the table on first call if it doesn't exist yet.
    """
    from delta.tables import DeltaTable

    path = f"{SILVER_CONTAINER_URL}/{name}/"
    full_name = silver_table(name)

    if not DeltaTable.isDeltaTable(spark, path):
        df.write.format("delta").mode("overwrite").save(path)
        spark.sql(f"CREATE TABLE IF NOT EXISTS {full_name} USING DELTA LOCATION '{path}'")
        return

    target = DeltaTable.forPath(spark, path)
    match_condition = " AND ".join([f"target.{c} = source.{c}" for c in merge_key_cols])

    (target.alias("target")
        .merge(df.alias("source"), match_condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())


def check_foreign_key(df: DataFrame, fk_col: str, ref_df: DataFrame, ref_col: str) -> tuple:
    """
    Returns (valid_df, invalid_df). invalid_df = rows where fk_col is
    populated but does NOT exist in ref_df's ref_col. Rows where fk_col is
    NULL pass through into valid_df untouched - a null FK isn't a broken
    reference, it's "nothing to check" (e.g. payment.company_id is
    legitimately null for CAPITAL_CALL/CONTRIBUTION types).
    """
    ref_keys = ref_df.select(F.col(ref_col).alias("_ref_key")).distinct()
    joined = df.join(ref_keys, df[fk_col] == F.col("_ref_key"), how="left")
    valid_df = (
        joined.filter(F.col("_ref_key").isNotNull() | F.col(fk_col).isNull())
              .drop("_ref_key")
    )
    invalid_df = (
        joined.filter(F.col("_ref_key").isNull() & F.col(fk_col).isNotNull())
              .drop("_ref_key")
              .withColumn("reason_code", F.lit("UNKNOWN_REFERENCE"))
    )
    return valid_df, invalid_df